In [5]:
import requests
import pandas as pd
from io import StringIO

url = "https://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

rmm = pd.read_csv(
    StringIO(response.text),
    sep=r"\s+",
    skiprows=2,
    header=None,
    names=[
        "year", "month", "day",
        "RMM1", "RMM2", "phase", "amplitude",
        "method"
    ],
    na_values=[1e36, "1.E36", 999],
)

rmm["date"] = pd.to_datetime(
    rmm[["year", "month", "day"]].astype(int)
)

rmm = (
    rmm
    .set_index("date")
    .drop(columns=["method"])
    .sort_index()
)

print(rmm.head())
print(rmm.tail())

            year  month  day     RMM1     RMM2  phase  amplitude
date                                                            
1974-06-01  1974      6    1  1.63447  1.20304    5.0    2.02948
1974-06-02  1974      6    2  1.60289  1.01512    5.0    1.89729
1974-06-03  1974      6    3  1.51625  1.08551    5.0    1.86476
1974-06-04  1974      6    4  1.50981  1.03573    5.0    1.83092
1974-06-05  1974      6    5  1.55906  1.30518    5.0    2.03326
            year  month  day      RMM1      RMM2  phase  amplitude
date                                                              
2024-02-20  2024      2   20 -0.305918  0.305419    8.0   0.432281
2024-02-21  2024      2   21  0.103033  0.071931    5.0   0.125658
2024-02-22  2024      2   22  0.236122  0.248544    6.0   0.342823
2024-02-23  2024      2   23  0.201856  0.325106    6.0   0.382675
2024-02-24  2024      2   24 -0.003146  0.128460    7.0   0.128498


In [7]:
rmm.to_csv("data/input/pacific/MJO.csv")

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# Load ENSO / ONI data
# ============================================================

ENSO_PATH = Path("data/input/pacific/ENSO.csv")

enso_raw = pd.read_csv(ENSO_PATH)

# Handles both cases:
# 1) columns: Date, ONI...
# 2) columns: unnamed index, Date, ONI...
if enso_raw.columns[0].lower().startswith("unnamed"):
    enso_raw = enso_raw.drop(columns=enso_raw.columns[0])

enso_raw = enso_raw.iloc[:, :2].copy()
enso_raw.columns = ["date", "ONI"]

enso_raw["date"] = pd.to_datetime(enso_raw["date"], errors="coerce")
enso_raw["ONI"] = pd.to_numeric(enso_raw["ONI"], errors="coerce")

# NOAA missing values
enso_raw["ONI"] = enso_raw["ONI"].replace([-99.9, -9999.0], np.nan)

enso = (
    enso_raw
    .dropna(subset=["date", "ONI"])
    .set_index("date")
    .sort_index()
)

# ============================================================
# Create useful modelling variables
# ============================================================

enso["ONI_lag1"] = enso["ONI"].shift(1)
enso["ONI_lag3"] = enso["ONI"].shift(3)
enso["ONI_lag6"] = enso["ONI"].shift(6)

enso["ONI_roll3"] = enso["ONI"].rolling(3).mean()
enso["ONI_roll6"] = enso["ONI"].rolling(6).mean()

enso["el_nino"] = (enso["ONI"] >= 0.5).astype(int)
enso["la_nina"] = (enso["ONI"] <= -0.5).astype(int)
enso["neutral"] = ((enso["ONI"] > -0.5) & (enso["ONI"] < 0.5)).astype(int)

print(enso.head())
print(enso.tail())

# Optional: save cleaned version
OUTPUT_PATH = Path("data/processed/pacific/ENSO_clean.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

enso.to_csv(OUTPUT_PATH)

print(f"Saved cleaned ENSO data to: {OUTPUT_PATH}")

             ONI  ONI_lag1  ONI_lag3  ONI_lag6  ONI_roll3  ONI_roll6  el_nino  \
date                                                                            
1950-01-01 -1.53       NaN       NaN       NaN        NaN        NaN        0   
1950-02-01 -1.34     -1.53       NaN       NaN        NaN        NaN        0   
1950-03-01 -1.16     -1.34       NaN       NaN  -1.343333        NaN        0   
1950-04-01 -1.18     -1.16     -1.53       NaN  -1.226667        NaN        0   
1950-05-01 -1.07     -1.18     -1.34       NaN  -1.136667        NaN        0   

            la_nina  neutral  
date                          
1950-01-01        1        0  
1950-02-01        1        0  
1950-03-01        1        0  
1950-04-01        1        0  
1950-05-01        1        0  
             ONI  ONI_lag1  ONI_lag3  ONI_lag6  ONI_roll3  ONI_roll6  el_nino  \
date                                                                            
2025-11-01 -0.55     -0.51     -0.28     -0.02  -0.48

In [9]:
enso

,Date,ONI from CPC missing value -99.9 https://psl.noaa.gov/data/timeseries/month/
0,1950-01-01,-1.53
1,1950-02-01,-1.34
2,1950-03-01,-1.16
3,1950-04-01,-1.18
4,1950-05-01,-1.07
...,...,...
919,2026-08-01,-9999.00
920,2026-09-01,-9999.00
921,2026-10-01,-9999.00
922,2026-11-01,-9999.00


In [12]:
pd.read_csv("data/input/pacific/Nino34.csv")

,Date,NINA34 missing value -99.99 https://psl.noaa.gov/data/timeseries/month/
0,1870-01-01,-1.00
1,1870-02-01,-1.20
2,1870-03-01,-0.83
3,1870-04-01,-0.81
4,1870-05-01,-1.27
...,...,...
1867,2025-08-01,-0.11
1868,2025-09-01,-0.30
1869,2025-10-01,-0.50
1870,2025-11-01,-0.68


In [14]:
from pathlib import Path
import json
import requests
import numpy as np
import pandas as pd


URL = "https://climatereanalyzer.org/clim/sst_daily/json_2clim/oisst2.1_nino3.4_sst_day.json"

RAW_PATH = Path("data/input/pacific/nino34_daily_raw.json")
OUT_PATH = Path("data/processed/pacific/NINO34_daily.csv")

RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# Download JSON
# ============================================================

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()

RAW_PATH.write_text(response.text, encoding="utf-8")

data = response.json()


# ============================================================
# Parse yearly daily data
# ============================================================

rows = []

for item in data:
    name = str(item["name"])

    # Keep only actual years: 1981, 1982, ...
    # Skip climatology entries like "1991-2020"
    if not name.isdigit():
        continue

    year = int(name)
    values = item["data"]

    dates = pd.date_range(
        start=f"{year}-01-01",
        periods=len(values),
        freq="D"
    )

    for date, value in zip(dates, values):
        rows.append(
            {
                "date": date,
                "nino34_sst": value,
                "source": item.get("data_source", None),
            }
        )

nino34 = pd.DataFrame(rows)


# ============================================================
# Clean
# ============================================================

nino34["date"] = pd.to_datetime(nino34["date"])
nino34["nino34_sst"] = pd.to_numeric(nino34["nino34_sst"], errors="coerce")

nino34 = (
    nino34
    .dropna(subset=["nino34_sst"])
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .set_index("date")
)

# Optional: remove source column if you only want numeric model input
nino34 = nino34[["nino34_sst"]]


# ============================================================
# Save
# ============================================================

nino34.to_csv(OUT_PATH)

print(nino34.head())
print(nino34.tail())
print("Date range:", nino34.index.min(), "to", nino34.index.max())
print("Saved to:", OUT_PATH)

            nino34_sst
date                  
1981-09-01      26.647
1981-09-02      26.633
1981-09-03      26.587
1981-09-04      26.396
1981-09-05      26.339
            nino34_sst
date                  
2026-05-19      28.835
2026-05-20      28.800
2026-05-21      28.777
2026-05-22      28.740
2026-05-23      28.731
Date range: 1981-09-01 00:00:00 to 2026-05-23 00:00:00
Saved to: data\processed\pacific\NINO34_daily.csv


In [15]:
nino34

,nino34_sst
date,
1981-09-01,26.647
1981-09-02,26.633
1981-09-03,26.587
1981-09-04,26.396
1981-09-05,26.339
...,...
2026-05-19,28.835
2026-05-20,28.800
2026-05-21,28.777


## ERA5

In [2]:
!pip install cdsapi xarray netcdf4 pandas numpy tqdm

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ---------------------- ----------------- 0.8/1.4 MB 932.1 kB/s eta 0:00:01
   ----------------------------- ---------- 1.0/1.4 MB 952.1 kB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.2 MB/s  0:00:02
   ---------------------------------------- 0.0/21.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/21.3 MB ? eta -:--:--
   - -----


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from pathlib import Path

cdsapirc_path = Path.home() / ".cdsapirc"

cds_token = "PASTE_YOUR_TOKEN_HERE"

cdsapirc_text = f"""
url: https://cds.climate.copernicus.eu/api
key: 80b8bbd8-726b-4e75-9e4e-706ba096dc03
""".strip()

cdsapirc_path.write_text(cdsapirc_text)

print(f"Saved CDS API config to: {cdsapirc_path}")

Saved CDS API config to: C:\Users\ilang\.cdsapirc


In [7]:
stations = {
    "BELO_HORIZONTE": {
        "lat": -19.883889,
        "lon": -43.969444,
        "source_note": "INMET Belo Horizonte Pampulha A521"
    },
    "CRUZEIRO_DO_SUL": {
        "lat": -7.610833,
        "lon": -72.681389,
        "source_note": "INMET Cruzeiro do Sul"
    },
    "DARWIN_AIRPORT": {
        "lat": -12.42,
        "lon": 130.89,
        "source_note": "BOM Darwin Airport 014015"
    },
    "GARANHUNS": {
        "lat": -8.910833,
        "lon": -36.493333,
        "source_note": "INMET Garanhuns A322"
    },
    "MANAUS": {
        "lat": -3.103333,
        "lon": -60.016389,
        "source_note": "INMET Manaus A101"
    },
    "SALVADOR": {
        "lat": -13.005833,
        "lon": -38.505833,
        "source_note": "INMET Salvador A401 approximate; verify in INMET catalogue"
    },
    "SAO_PAULO": {
        "lat": -23.496294,
        "lon": -46.620088,
        "source_note": "INMET Sao Paulo Mirante de Santana A701 approximate; verify in INMET catalogue"
    },
    "TORONTO": {
        "lat": 43.68,
        "lon": -79.63,
        "source_note": "Environment Canada Toronto Pearson"
    },
}

In [22]:
import cdsapi
import pandas as pd
import xarray as xr
import numpy as np
from pathlib import Path
from tqdm import tqdm

BASE_DIR = Path("data/input/ERA5")
TEMP_DIR = BASE_DIR / "temperature"
HUMIDITY_DIR = BASE_DIR / "humidity"
RAW_DIR = BASE_DIR / "raw_daily"

TEMP_DIR.mkdir(parents=True, exist_ok=True)
HUMIDITY_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

client = cdsapi.Client()

YEARS = list(range(2006, 2026))

def round_to_era5_grid(x, grid=0.25):
    return round(x / grid) * grid

def era5_single_cell_area(lat, lon):
    """
    ERA5 grid is 0.25 degrees.
    This creates a tiny box exactly around the nearest ERA5 grid point.

    CDS area format: [North, West, South, East]
    """
    grid_lat = round_to_era5_grid(lat, 0.25)
    grid_lon = round_to_era5_grid(lon, 0.25)

    return [
        grid_lat + 0.125,
        grid_lon - 0.125,
        grid_lat - 0.125,
        grid_lon + 0.125,
    ]

def download_station_month(station_name, lat, lon, year, month):
    target = RAW_DIR / f"{station_name}_{year}_{month:02d}_era5_daily.nc"

    if target.exists():
        print(f"Already exists: {target}")
        return target

    request = {
        "product_type": "reanalysis",
        "variable": [
            "2m_temperature",
            "2m_dewpoint_temperature",
        ],
        "year": str(year),
        "month": f"{month:02d}",
        "day": [f"{d:02d}" for d in range(1, 32)],
        "daily_statistic": "daily_mean",
        "time_zone": "UTC+00:00",
        "frequency": "1_hourly",
        "area": era5_single_cell_area(lat, lon),
        "data_format": "netcdf",
        "download_format": "unarchived",
    }

    print(f"Downloading DAILY {station_name} {year}-{month:02d}...")

    client.retrieve(
        "derived-era5-single-levels-daily-statistics",
        request,
        str(target)
    )

    return target

2026-06-13 14:56:38,836 INFO [2026-06-11T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 15 June. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure-part-2/150414).


In [35]:
import zipfile
import tempfile
from pathlib import Path
import pandas as pd
import xarray as xr

SAFE_EXTRACT_BASE = Path(tempfile.gettempdir()) / "era5_extract"
SAFE_EXTRACT_BASE.mkdir(parents=True, exist_ok=True)


def unzip_if_needed(path):
    path = Path(path)

    extract_dir = SAFE_EXTRACT_BASE / path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)

    with open(path, "rb") as f:
        signature = f.read(2)

    if signature == b"PK":
        with zipfile.ZipFile(path, "r") as z:
            z.extractall(extract_dir)
    else:
        raise ValueError(f"Expected ZIP file, but got non-ZIP: {path}")

    return extract_dir


def find_temp_and_dewpoint_files(extract_dir):
    extract_dir = Path(extract_dir)

    nc_files = list(extract_dir.glob("*.nc"))

    print("Files found:")
    for f in nc_files:
        print(f, "exists:", f.exists())

    temp_files = [
        f for f in nc_files
        if "2m_temperature" in f.name and "dewpoint" not in f.name
    ]

    dew_files = [
        f for f in nc_files
        if "2m_dewpoint" in f.name
    ]

    if len(temp_files) != 1:
        raise FileNotFoundError(f"Expected 1 temperature file, found {len(temp_files)}: {temp_files}")

    if len(dew_files) != 1:
        raise FileNotFoundError(f"Expected 1 dewpoint file, found {len(dew_files)}: {dew_files}")

    return temp_files[0], dew_files[0]


def get_time_column(ds):
    for candidate in ["valid_time", "time", "date"]:
        if candidate in ds.coords:
            return candidate
    for candidate in ["valid_time", "time", "date"]:
        if candidate in ds.variables:
            return candidate
    raise ValueError(f"No time coordinate found. Available coords: {list(ds.coords)}")


def dataset_to_daily_df(file_path, lat, lon, variable_name, output_name):
    file_path = str(Path(file_path).resolve())

    ds = xr.open_dataset(file_path, engine="netcdf4")

    time_col = get_time_column(ds)

    target_lon = lon
    if float(ds.longitude.max()) > 180 and lon < 0:
        target_lon = lon % 360

    point = ds.sel(
        latitude=lat,
        longitude=target_lon,
        method="nearest"
    )

    df = point[[variable_name]].to_dataframe().reset_index()
    ds.close()

    df = df[[time_col, variable_name]].rename(
        columns={
            time_col: "date",
            variable_name: output_name
        }
    )

    df["date"] = pd.to_datetime(df["date"])
    df[output_name] = df[output_name] - 273.15

    return df


def extract_nearest_daily(nc_path, station_name, lat, lon):
    extract_dir = unzip_if_needed(nc_path)

    temp_file, dew_file = find_temp_and_dewpoint_files(extract_dir)

    temp_df = dataset_to_daily_df(
        file_path=temp_file,
        lat=lat,
        lon=lon,
        variable_name="t2m",
        output_name="temperature_2m_c"
    )

    dew_df = dataset_to_daily_df(
        file_path=dew_file,
        lat=lat,
        lon=lon,
        variable_name="d2m",
        output_name="dewpoint_2m_c"
    )

    daily = pd.merge(temp_df, dew_df, on="date", how="inner")

    daily["station"] = station_name
    daily["station_lat"] = lat
    daily["station_lon"] = lon

    return daily[[
        "date",
        "station",
        "station_lat",
        "station_lon",
        "temperature_2m_c",
        "dewpoint_2m_c"
    ]]


def save_daily_files(station_name, daily):
    temp = daily[[
        "date",
        "station",
        "station_lat",
        "station_lon",
        "temperature_2m_c"
    ]].copy()

    hum = daily[[
        "date",
        "station",
        "station_lat",
        "station_lon",
        "dewpoint_2m_c"
    ]].copy()

    temp_path = TEMP_DIR / f"{station_name}_ERA5_temperature_2m_daily_2006_2025.csv"
    hum_path = HUMIDITY_DIR / f"{station_name}_ERA5_dewpoint_2m_daily_2006_2025.csv"

    temp.to_csv(temp_path, index=False)
    hum.to_csv(hum_path, index=False)

    print(f"Saved: {temp_path}")
    print(f"Saved: {hum_path}")

In [ ]:
all_daily = []

for station_name, meta in stations.items():
    lat = meta["lat"]
    lon = meta["lon"]

    monthly_daily = []

    for year in YEARS:
        for month in range(1, 13):
            nc_path = download_station_month(
                station_name=station_name,
                lat=lat,
                lon=lon,
                year=year,
                month=month
            )

            daily = extract_nearest_daily(
                nc_path=nc_path,
                station_name=station_name,
                lat=lat,
                lon=lon
            )

            monthly_daily.append(daily)

    station_daily = (
        pd.concat(monthly_daily, ignore_index=True)
          .drop_duplicates(subset=["date", "station"])
          .sort_values("date")
          .reset_index(drop=True)
    )

    save_daily_files(station_name, station_daily)
    all_daily.append(station_daily)

combined = pd.concat(all_daily, ignore_index=True)

combined_path = BASE_DIR / "ERA5_temperature_dewpoint_daily_all_stations_2006_2025.csv"
combined.to_csv(combined_path, index=False)

print(f"Saved combined file: {combined_path}")

combined.head()

Already exists: data\input\ERA5\raw_daily\BELO_HORIZONTE_2006_01_era5_daily.nc
Files found:
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2006_01_era5_daily\2m_dewpoint_temperature_0_daily-mean.nc exists: True
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2006_01_era5_daily\2m_temperature_stream-oper_daily-mean.nc exists: True


2026-06-13 15:06:44,420 WARNING [2026-06-08T00:00:00] An issue with the following parameters has been identified:

- `maximum_2m_temperature_since_previous_post_processing`
- `minimum_2m_temperature_since_previous_post_processing`
- `10m_wind_gust_since_previous_post_processing`
- `maximum_total_precipitation_rate_since_previous_post_processing`
- `minimum_total_precipitation_rate_since_previous_post_processing`

Please refer to the documented [Known issue](https://confluence.ecmwf.int/display/CKB/ERA5+family+post-processed+daily+statistics+documentation#ERA5familypostprocesseddailystatisticsdocumentation-KnownIssues) for details. **Data downloaded for the parameters listed above should not be used.** A fix is to be released as soon as possible. Please [follow updates on this topic on our Forum announcement](https://forum.ecmwf.int/t/issue-affecting-some-parameters-from-the-era5-post-processed-daily-statistics-on-single-levels/15057).
2026-06-13 15:06:44,421 INFO Request ID is c781a038

b0024c9c20ff3607fcea5eaf8e245f70.zip:   0%|          | 0.00/49.5k [00:00<?, ?B/s]

Files found:
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2006_02_era5_daily\2m_dewpoint_temperature_0_daily-mean.nc exists: True
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2006_02_era5_daily\2m_temperature_stream-oper_daily-mean.nc exists: True


2026-06-13 15:08:48,895 WARNING [2026-06-08T00:00:00] An issue with the following parameters has been identified:

- `maximum_2m_temperature_since_previous_post_processing`
- `minimum_2m_temperature_since_previous_post_processing`
- `10m_wind_gust_since_previous_post_processing`
- `maximum_total_precipitation_rate_since_previous_post_processing`
- `minimum_total_precipitation_rate_since_previous_post_processing`

Please refer to the documented [Known issue](https://confluence.ecmwf.int/display/CKB/ERA5+family+post-processed+daily+statistics+documentation#ERA5familypostprocesseddailystatisticsdocumentation-KnownIssues) for details. **Data downloaded for the parameters listed above should not be used.** A fix is to be released as soon as possible. Please [follow updates on this topic on our Forum announcement](https://forum.ecmwf.int/t/issue-affecting-some-parameters-from-the-era5-post-processed-daily-statistics-on-single-levels/15057).
2026-06-13 15:08:48,896 INFO Request ID is 1523aa63

...

Files found:
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2015_10_era5_daily\2m_dewpoint_temperature_0_daily-mean.nc exists: True
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2015_10_era5_daily\2m_temperature_stream-oper_daily-mean.nc exists: True


2026-06-14 05:12:38,064 WARNING [2026-06-08T00:00:00] An issue with the following parameters has been identified:

- `maximum_2m_temperature_since_previous_post_processing`
- `minimum_2m_temperature_since_previous_post_processing`
- `10m_wind_gust_since_previous_post_processing`
- `maximum_total_precipitation_rate_since_previous_post_processing`
- `minimum_total_precipitation_rate_since_previous_post_processing`

Please refer to the documented [Known issue](https://confluence.ecmwf.int/display/CKB/ERA5+family+post-processed+daily+statistics+documentation#ERA5familypostprocesseddailystatisticsdocumentation-KnownIssues) for details. **Data downloaded for the parameters listed above should not be used.** A fix is to be released as soon as possible. Please [follow updates on this topic on our Forum announcement](https://forum.ecmwf.int/t/issue-affecting-some-parameters-from-the-era5-post-processed-daily-statistics-on-single-levels/15057).
2026-06-14 05:12:38,065 INFO Request ID is e4001d36

ba1237a8f77d8c33965db4a12581617e.zip:   0%|          | 0.00/49.5k [00:00<?, ?B/s]

Files found:
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2015_11_era5_daily\2m_dewpoint_temperature_0_daily-mean.nc exists: True
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2015_11_era5_daily\2m_temperature_stream-oper_daily-mean.nc exists: True


2026-06-14 05:57:47,181 WARNING [2026-06-08T00:00:00] An issue with the following parameters has been identified:

- `maximum_2m_temperature_since_previous_post_processing`
- `minimum_2m_temperature_since_previous_post_processing`
- `10m_wind_gust_since_previous_post_processing`
- `maximum_total_precipitation_rate_since_previous_post_processing`
- `minimum_total_precipitation_rate_since_previous_post_processing`

Please refer to the documented [Known issue](https://confluence.ecmwf.int/display/CKB/ERA5+family+post-processed+daily+statistics+documentation#ERA5familypostprocesseddailystatisticsdocumentation-KnownIssues) for details. **Data downloaded for the parameters listed above should not be used.** A fix is to be released as soon as possible. Please [follow updates on this topic on our Forum announcement](https://forum.ecmwf.int/t/issue-affecting-some-parameters-from-the-era5-post-processed-daily-statistics-on-single-levels/15057).
2026-06-14 05:57:47,182 INFO Request ID is e1532e70

In [36]:
daily = extract_nearest_daily(
    "data/input/ERA5/raw_daily/BELO_HORIZONTE_2006_01_era5_daily.nc",
    "BELO_HORIZONTE",
    -19.883889,
    -43.969444
)

daily.head()

Files found:
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2006_01_era5_daily\2m_dewpoint_temperature_0_daily-mean.nc exists: True
C:\Users\ilang\AppData\Local\Temp\era5_extract\BELO_HORIZONTE_2006_01_era5_daily\2m_temperature_stream-oper_daily-mean.nc exists: True


,date,station,station_lat,station_lon,temperature_2m_c,dewpoint_2m_c
0,2006-01-01,BELO_HORIZONTE,-19.883889,-43.969444,22.339172,17.851685
1,2006-01-02,BELO_HORIZONTE,-19.883889,-43.969444,21.696472,17.959106
2,2006-01-03,BELO_HORIZONTE,-19.883889,-43.969444,21.860992,18.134094
3,2006-01-04,BELO_HORIZONTE,-19.883889,-43.969444,22.407471,17.738892
4,2006-01-05,BELO_HORIZONTE,-19.883889,-43.969444,21.219299,17.531799


In [26]:
test_file = files[0]

with open(test_file, "rb") as f:
    print(f.read(20))

b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x03&\xcd\\\xa8t\xccnfb'


## Yearly ERA5

In [2]:
from pathlib import Path
import cdsapi
import pandas as pd

BASE_DIR = Path("data/input/ERA5")
TEMP_DIR = BASE_DIR / "temperature"
HUMIDITY_DIR = BASE_DIR / "humidity"
RAW_DIR = BASE_DIR / "raw_timeseries"

TEMP_DIR.mkdir(parents=True, exist_ok=True)
HUMIDITY_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

client = cdsapi.Client()

START_DATE = "2015-12-01"
END_DATE = "2025-12-31"

def download_station_timeseries(station_name, lat, lon):
    target = RAW_DIR / f"{station_name}_ERA5_timeseries_2015_12_2025_12.csv"

    if target.exists():
        print(f"Already exists: {target}")
        return target

    request = {
        "variable": [
            "2m_temperature",
            "2m_dewpoint_temperature",
        ],
        "location": {
            "latitude": lat,
            "longitude": lon,
        },
        "date": [f"{START_DATE}/{END_DATE}"],
        "data_format": "csv",
    }

    print(f"Downloading ERA5 time-series for {station_name}...")

    client.retrieve(
        "reanalysis-era5-single-levels-timeseries",
        request,
        str(target)
    )

    return target

2026-06-14 06:36:32,756 INFO [2026-06-11T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 15 June. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure-part-2/150414).


In [3]:
def process_timeseries_csv(csv_path, station_name, lat, lon):
    df = pd.read_csv(csv_path)

    print("Columns:", df.columns.tolist())
    print(df.head())

    # Try to detect time column
    possible_time_cols = ["time", "valid_time", "date", "datetime"]
    time_col = None

    for col in possible_time_cols:
        if col in df.columns:
            time_col = col
            break

    if time_col is None:
        raise ValueError(f"Could not find time column. Columns are: {df.columns.tolist()}")

    df[time_col] = pd.to_datetime(df[time_col])
    df["date"] = df[time_col].dt.date

    # Try to detect variable columns
    temp_col = None
    dew_col = None

    for col in df.columns:
        low = col.lower()
        if "2m_temperature" in low or low == "t2m":
            temp_col = col
        if "2m_dewpoint" in low or low == "d2m":
            dew_col = col

    if temp_col is None:
        raise ValueError(f"Could not find temperature column. Columns are: {df.columns.tolist()}")

    if dew_col is None:
        raise ValueError(f"Could not find dewpoint column. Columns are: {df.columns.tolist()}")

    daily = (
        df.groupby("date", as_index=False)
          .agg(
              temperature_2m_c=(temp_col, "mean"),
              dewpoint_2m_c=(dew_col, "mean"),
          )
    )

    # ERA5 likely comes in Kelvin
    if daily["temperature_2m_c"].mean() > 100:
        daily["temperature_2m_c"] = daily["temperature_2m_c"] - 273.15

    if daily["dewpoint_2m_c"].mean() > 100:
        daily["dewpoint_2m_c"] = daily["dewpoint_2m_c"] - 273.15

    daily["date"] = pd.to_datetime(daily["date"])
    daily["station"] = station_name
    daily["station_lat"] = lat
    daily["station_lon"] = lon

    return daily[[
        "date",
        "station",
        "station_lat",
        "station_lon",
        "temperature_2m_c",
        "dewpoint_2m_c",
    ]]


def save_station_files(station_name, daily):
    temp_path = TEMP_DIR / f"{station_name}_ERA5_temperature_2m_daily_2015_12_2025_12.csv"
    hum_path = HUMIDITY_DIR / f"{station_name}_ERA5_dewpoint_2m_daily_2015_12_2025_12.csv"

    daily[[
        "date", "station", "station_lat", "station_lon", "temperature_2m_c"
    ]].to_csv(temp_path, index=False)

    daily[[
        "date", "station", "station_lat", "station_lon", "dewpoint_2m_c"
    ]].to_csv(hum_path, index=False)

    print(f"Saved: {temp_path}")
    print(f"Saved: {hum_path}")

In [8]:
stations_to_run = {
    "BELO_HORIZONTE": stations["BELO_HORIZONTE"]
}

all_daily = []

for station_name, meta in stations_to_run.items():
    lat = meta["lat"]
    lon = meta["lon"]

    csv_path = download_station_timeseries(station_name, lat, lon)

    daily = process_timeseries_csv(csv_path, station_name, lat, lon)

    save_station_files(station_name, daily)
    all_daily.append(daily)

combined = pd.concat(all_daily, ignore_index=True)

combined_path = BASE_DIR / "ERA5_temperature_dewpoint_daily_BELO_HORIZONTE_2015_12_2025_12.csv"
combined.to_csv(combined_path, index=False)

combined.head()

2026-06-14 06:37:27,949 INFO [2026-02-16T00:00:00] - To generate this ERA5 hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/R6cfHg) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202).
2026-06-14 06:37:27,950 INFO Request ID is 88d3f870-4859-4c3f-9727-67274fb11217
2026-06-14 06:37:28,361 INF

8268ee4ae12b3cc6fbacd7a7c317604b.zip:   0%|          | 0.00/968k [00:00<?, ?B/s]

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf5 in position 10: invalid start byte